# 🛡️ BGT-1009 Programlamaya Giriş
## 08. Hafta: Dosyalar ve Hata Yönetimi (Google Colab Uyumlu)

Bu defter ÇOMÜ Bilgisayar Güvenliği Programlamaya Giriş dersi 8. hafta uygulamalarını içerir.
- Tüm örnekler **bağımsızdır** ve kendi sentetik verisini otomatik oluşturur.
- `pathlib.Path` kullanıldığı için Google Colab, Jupyter Notebook ve yerel ortamlarda sıfır ayarla çalışır.

[Ders web sitesine git](https://cyasar.github.io/prg/#/hafta/8/sunum/1)

### 📌 Örnek: csv_read_events.py (Csv Read Events)

In [ ]:
# CSV okuma: csv.DictReader başlık satırını sözlük anahtarı olarak kullanır
import csv
from pathlib import Path

klasor = Path("hafta08_calisma")
klasor.mkdir(exist_ok=True)
olay_dosyasi = klasor / "olaylar.csv"

# Sentetik (kurgusal) olay verisi. Gerçek kişisel veri kullanmıyoruz.
satirlar = [
    ["zaman", "kullanici", "ip", "olay", "boyut_kb"],
    ["08:01", "ayse", "10.0.0.5", "LOGIN_OK", "2"],
    ["08:03", "mehmet", "10.0.0.8", "LOGIN_FAIL", "1"],
    ["08:04", "mehmet", "10.0.0.8", "LOGIN_FAIL", "1"],
    ["08:10", "zeynep", "10.0.0.12", "FILE_UPLOAD", "850"],
    ["08:15", "mehmet", "10.0.0.8", "LOGIN_FAIL", "1"],
    ["08:20", "ayse", "10.0.0.5", "FILE_UPLOAD", "120"],
]
# newline="" CSV modülünün satır sonlarını kendisinin yönetmesi için gereklidir.
with open(olay_dosyasi, "w", encoding="utf-8", newline="") as f:
    yazici = csv.writer(f)
    yazici.writerows(satirlar)

basarisiz = 0
toplam_kb = 0
with open(olay_dosyasi, encoding="utf-8", newline="") as f:
    okuyucu = csv.DictReader(f)
    print("Sütunlar:", okuyucu.fieldnames)
    for kayit in okuyucu:
        boyut = int(kayit["boyut_kb"])   # CSV'deki her değer metindir -> int()
        toplam_kb += boyut
        if kayit["olay"] == "LOGIN_FAIL":
            basarisiz += 1
        print(f"{kayit['zaman']} {kayit['kullanici']:<7} {kayit['olay']:<12} {boyut:>4} KB")

print("Başarısız giriş sayısı:", basarisiz)
print("Toplam veri:", toplam_kb, "KB")


### 📌 Örnek: csv_summary_report.py (Csv Summary Report)

In [ ]:
# Haftanın uygulaması: sentetik CSV günlüğünden kullanıcı bazlı ÖZET dosyası üretme
import csv
from pathlib import Path

klasor = Path("hafta08_calisma")
klasor.mkdir(exist_ok=True)
girdi = klasor / "gunluk.csv"
cikti = klasor / "ozet.csv"

ornek_veri = """zaman,kullanici,olay
08:01,ayse,LOGIN_OK
08:03,mehmet,LOGIN_FAIL
08:04,mehmet,LOGIN_FAIL
08:05,mehmet,LOGIN_FAIL
08:07,zeynep,LOGIN_FAIL
08:09,zeynep,LOGIN_OK
08:12,ayse,LOGOUT
"""
girdi.write_text(ornek_veri, encoding="utf-8")

# 1) Oku ve kullanıcı başına say
ozet = {}
with open(girdi, encoding="utf-8", newline="") as f:
    for kayit in csv.DictReader(f):
        kullanici = kayit["kullanici"]
        if kullanici not in ozet:
            ozet[kullanici] = {"toplam": 0, "basarisiz": 0}
        ozet[kullanici]["toplam"] += 1
        if kayit["olay"] == "LOGIN_FAIL":
            ozet[kullanici]["basarisiz"] += 1

# 2) Özeti yeni bir CSV dosyasına yaz
ESIK = 3
with open(cikti, "w", encoding="utf-8", newline="") as f:
    alanlar = ["kullanici", "toplam_olay", "basarisiz", "durum"]
    yazici = csv.DictWriter(f, fieldnames=alanlar)
    yazici.writeheader()
    for kullanici in sorted(ozet):
        bilgi = ozet[kullanici]
        durum = "INCELE" if bilgi["basarisiz"] >= ESIK else "NORMAL"
        yazici.writerow({
            "kullanici": kullanici,
            "toplam_olay": bilgi["toplam"],
            "basarisiz": bilgi["basarisiz"],
            "durum": durum,
        })

# 3) Üretilen dosyayı doğrula
print("Özet dosyası oluşturuldu:", cikti.as_posix())
print(cikti.read_text(encoding="utf-8"), end="")


### 📌 Örnek: env_detect_colab.py (Env Detect Colab)

In [ ]:
# Hangi ortamda çalışıyorum? try/except ImportError ile ortam tespiti
import sys
from pathlib import Path

try:
    import google.colab  # Bu modül yalnızca Google Colab'da bulunur
    ortam = "Google Colab"
except ImportError:
    if sys.platform == "emscripten":
        ortam = "Tarayıcı (Pyodide - ders sitesi)"
    else:
        ortam = "Yerel bilgisayar (VS Code, IDLE, Terminal veya Jupyter)"

print("Çalışma ortamı:", ortam)

deneme = Path("hafta08_calisma") / "yazma_testi.txt"
deneme.parent.mkdir(exist_ok=True)
deneme.write_text("merhaba\n", encoding="utf-8")
print("Yazma testi:", "başarılı" if deneme.exists() else "başarısız")

if ortam == "Google Colab":
    print("İpucu: Sol menüdeki klasör simgesinden hafta08_calisma klasörünü gör.")
    print("Kendi dosyanı yüklemek için: from google.colab import files; files.upload()")
elif ortam.startswith("Tarayıcı"):
    print("İpucu: Dosyalar tarayıcı belleğindedir; sayfa yenilenince silinir.")
else:
    print("İpucu: Dosyalar, programı çalıştırdığın klasörün içinde oluşturulur.")


### 📌 Örnek: exception_types.py (Exception Types)

In [ ]:
# Sık karşılaşılan hata (exception) türlerini tanıyalım ve kontrollü yakalayalım
portlar = [22, 80, 443]
kullanicilar = {"ayse": "admin", "mehmet": "analist"}

print("1) ValueError: metin sayıya çevrilemiyor")
try:
    sayi = int("on iki")
except ValueError as hata:
    print("   Yakalandı:", hata)

print("2) ZeroDivisionError: sıfıra bölme")
try:
    oran = 10 / 0
except ZeroDivisionError as hata:
    print("   Yakalandı:", hata)

print("3) IndexError: listede olmayan indeks")
try:
    print(portlar[5])
except IndexError as hata:
    print("   Yakalandı:", hata)

print("4) KeyError: sözlükte olmayan anahtar")
try:
    rol = kullanicilar["zeynep"]
except KeyError as hata:
    print("   Yakalandı: anahtar bulunamadı ->", hata)

print("5) TypeError: uyumsuz türler")
try:
    mesaj = "Port: " + 443
except TypeError as hata:
    print("   Yakalandı:", hata)

print("6) FileNotFoundError: olmayan dosya")
try:
    with open("hafta08_calisma/olmayan_dosya.txt", encoding="utf-8") as f:
        f.read()
except FileNotFoundError:
    print("   Yakalandı: dosya bulunamadı")

print("Program çökmeden sona ulaştı.")


### 📌 Örnek: file_append_audit.py (File Append Audit)

In [ ]:
# Ekleme modu ("a"): mevcut içeriği silmeden dosyanın sonuna yazar
# Güvenlikte denetim kaydı (audit log) tam olarak böyle tutulur.
from pathlib import Path

klasor = Path("hafta08_calisma")
klasor.mkdir(exist_ok=True)
denetim = klasor / "denetim.log"

# Her çalıştırmada aynı sonucu görmek için dosyayı önce "w" ile sıfırlıyoruz.
with open(denetim, "w", encoding="utf-8") as f:
    f.write("2026-11-10 09:00 | sistem | Denetim kaydı başlatıldı\n")

olaylar = [
    ("09:05", "ayse", "Oturum açtı"),
    ("09:12", "mehmet", "Parola değiştirdi"),
    ("09:30", "ayse", "Rapor indirdi"),
]

# "a" modu: her yeni olay dosyanın SONUNA eklenir, eski kayıtlar korunur.
for saat, kullanici, eylem in olaylar:
    with open(denetim, "a", encoding="utf-8") as f:
        f.write(f"2026-11-10 {saat} | {kullanici} | {eylem}\n")

with open(denetim, encoding="utf-8") as f:
    kayitlar = f.read().splitlines()

print("Denetim kaydı satır sayısı:", len(kayitlar))
for kayit in kayitlar:
    print(kayit)
print("Son eylem:", kayitlar[-1].split(" | ")[2])


### 📌 Örnek: file_not_found_safe.py (File Not Found Safe)

In [ ]:
# Dosya yoksa programı çökertmek yerine güvenli varsayılan ayarlara dönme
from pathlib import Path

VARSAYILAN = {"max_deneme": "3", "kilit_suresi_dk": "15"}


def ayar_yukle(yol):
    ayarlar = dict(VARSAYILAN)           # varsayılanların kopyası
    try:
        with open(yol, encoding="utf-8") as f:
            for satir in f:
                satir = satir.strip()
                if not satir or satir.startswith("#"):
                    continue             # boş satır ve yorumları atla
                anahtar, deger = satir.split("=")
                ayarlar[anahtar.strip()] = deger.strip()
    except FileNotFoundError:
        print(f"[UYARI] {yol.name} bulunamadı, varsayılan ayarlar kullanılıyor.")
    return ayarlar


klasor = Path("hafta08_calisma")
klasor.mkdir(exist_ok=True)
ayar_dosyasi = klasor / "guvenlik.conf"
ayar_dosyasi.unlink(missing_ok=True)     # örnek her seferinde dosyasız başlasın

print("1. deneme:", ayar_yukle(ayar_dosyasi))

ayar_dosyasi.write_text(
    "# Güvenlik ayarları\nmax_deneme = 5\n\nkilit_suresi_dk = 30\n",
    encoding="utf-8",
)
print("2. deneme:", ayar_yukle(ayar_dosyasi))


### 📌 Örnek: file_write_read.py (File Write Read)

In [ ]:
# Dosyaya yazma ve okuma: with open() + UTF-8
# Örnek önce kendi sentetik dosyasını oluşturur. Bu yüzden bilgisayarda,
# Google Colab'da, Jupyter'da ve ders sitesinde aynı şekilde çalışır.
from pathlib import Path

klasor = Path("hafta08_calisma")
klasor.mkdir(exist_ok=True)
dosya = klasor / "notlar.txt"

# 1) Yazma ("w"): dosya yoksa oluşturur, varsa içeriğini SİLİP baştan yazar.
with open(dosya, "w", encoding="utf-8") as f:
    f.write("Güvenlik duvarı kuralı güncellendi\n")
    f.write("Yedekleme tamamlandı\n")
    f.write("Şüpheli oturum incelendi\n")

# 2) read(): dosyanın tamamını tek bir metin olarak döndürür.
with open(dosya, "r", encoding="utf-8") as f:
    icerik = f.read()
print("--- read() ---")
print(icerik, end="")

# 3) Dosya nesnesi üzerinde for: satır satır okur (büyük dosyalarda bellek dostu).
print("--- for satir in f ---")
with open(dosya, "r", encoding="utf-8") as f:
    numara = 1
    for satir in f:
        print(f"{numara}. {satir.strip()}")
        numara += 1

# 4) readlines(): tüm satırları liste olarak verir (satır sonu \n dahil).
with open(dosya, "r", encoding="utf-8") as f:
    satirlar = f.readlines()
print("Satır sayısı:", len(satirlar))
print("İlk satır (repr):", repr(satirlar[0]))


### 📌 Örnek: log_file_analyzer.py (Log File Analyzer)

In [ ]:
# Log DOSYASI analizi: dosya okuma + string işlemleri + bozuk satırlara dayanıklılık
from pathlib import Path

klasor = Path("hafta08_calisma")
klasor.mkdir(exist_ok=True)
log_dosyasi = klasor / "auth.log"
rapor_dosyasi = klasor / "auth_rapor.txt"

# Sentetik auth logu: bir satır bilerek bozuk, bir satır boş.
log_dosyasi.write_text(
    "2026-11-10 08:00:01 OK 10.0.0.5 ayse\n"
    "2026-11-10 08:01:15 FAIL 203.0.113.7 admin\n"
    "2026-11-10 08:01:17 FAIL 203.0.113.7 admin\n"
    "BOZUK_SATIR_KAYIT\n"
    "\n"
    "2026-11-10 08:01:20 FAIL 203.0.113.7 root\n"
    "2026-11-10 08:02:00 FAIL 198.51.100.4 test\n"
    "2026-11-10 08:05:42 OK 10.0.0.8 mehmet\n",
    encoding="utf-8",
)

basarisiz_ip = {}
islenen = 0
atlanan = 0

try:
    with open(log_dosyasi, encoding="utf-8") as f:
        no = 0
        for satir in f:
            no += 1
            satir = satir.strip()
            if not satir:
                continue
            try:
                tarih, saat, durum, ip, kullanici = satir.split()
            except ValueError:
                atlanan += 1
                print(f"[ATLANDI] {no}. satır beklenen biçimde değil")
                continue
            islenen += 1
            if durum == "FAIL":
                # get(anahtar, 0): anahtar yoksa 0 döndürür, KeyError oluşmaz
                basarisiz_ip[ip] = basarisiz_ip.get(ip, 0) + 1
except FileNotFoundError:
    print("Log dosyası bulunamadı:", log_dosyasi.name)
else:
    with open(rapor_dosyasi, "w", encoding="utf-8") as rapor:
        rapor.write("AUTH LOG RAPORU\n")
        rapor.write(f"İşlenen satır : {islenen}\n")
        rapor.write(f"Atlanan satır : {atlanan}\n")
        for ip, adet in basarisiz_ip.items():
            etiket = "ŞÜPHELİ" if adet >= 3 else "izle"
            rapor.write(f"{ip:<15} {adet} başarısız  [{etiket}]\n")
    print(rapor_dosyasi.read_text(encoding="utf-8"), end="")


### 📌 Örnek: path_portable.py (Path Portable)

In [ ]:
# pathlib ile her ortamda çalışan (taşınabilir) dosya yolları
# Windows "C:\..." ile Linux/Colab "/content/..." farkını Path nesnesi yönetir.
from pathlib import Path

kok = Path("hafta08_calisma")
rapor_klasoru = kok / "raporlar"          # / operatörü yol parçalarını birleştirir
rapor_klasoru.mkdir(parents=True, exist_ok=True)

dosyalar = {
    "ozet.txt": "Toplam olay: 12\n",
    "uyarilar.txt": "Kritik: 2\nOrta: 5\n",
    "eski.bak": "yedek\n",
}
for ad, icerik in dosyalar.items():
    (rapor_klasoru / ad).write_text(icerik, encoding="utf-8")

hedef = rapor_klasoru / "uyarilar.txt"
print("Göreli yol     :", hedef.as_posix())
print("Dosya adı      :", hedef.name)
print("Uzantı         :", hedef.suffix)
print("Ad (uzantısız) :", hedef.stem)
print("Var mı?        :", hedef.exists())

print("Klasördeki .txt dosyaları:")
for yol in sorted(rapor_klasoru.glob("*.txt")):
    print("  -", yol.name)

eksik = rapor_klasoru / "yok.txt"
print("yok.txt var mı?:", eksik.exists())
print("Okunan satırlar:", hedef.read_text(encoding="utf-8").splitlines())


### 📌 Örnek: raise_validation.py (Raise Validation)

In [ ]:
# raise ile kendi doğrulama kuralını yazma ve anlaşılır hata mesajı verme
def kullanici_adi_dogrula(ad):
    if not isinstance(ad, str):
        raise TypeError("Kullanıcı adı metin olmalıdır.")
    ad = ad.strip()
    if ad == "":
        raise ValueError("Kullanıcı adı boş olamaz.")
    if not 3 <= len(ad) <= 16:
        raise ValueError("Kullanıcı adı 3-16 karakter olmalıdır.")
    if not ad.isalnum():
        raise ValueError("Yalnızca harf ve rakam kullanılabilir.")
    return ad.lower()


adaylar = ["Ayse01", "   ", "ab", "admin'--", 2026, "  Zeynep  "]
for aday in adaylar:
    try:
        temiz = kullanici_adi_dogrula(aday)
        print(f"KABUL : {aday!r:<12} -> {temiz}")
    except (ValueError, TypeError) as hata:
        print(f"RED   : {aday!r:<12} -> {hata}")


### 📌 Örnek: try_except_basics.py (Try Except Basics)

In [ ]:
# try / except / else / finally ile kullanıcı girdisini güvenle işleme
def port_oku(metin):
    try:
        port = int(metin)                # ValueError oluşabilir
    except ValueError:
        print(f"Hata: '{metin}' bir sayı değil.")
        return None
    else:                                # hata OLUŞMAZSA çalışır
        if not 1 <= port <= 65535:
            print(f"Hata: {port} geçerli port aralığında değil.")
            return None
        print(f"Kabul: {port}")
        return port
    finally:                             # her durumda çalışır (return olsa bile)
        print("  [kontrol tamamlandı]")


gecerli = []
for sira in range(1, 4):
    metin = input(f"{sira}. port: ")
    print()
    sonuc = port_oku(metin)
    if sonuc is not None:
        gecerli.append(sonuc)

print("Geçerli portlar:", gecerli)
